# E9 GNN Navigation

Author: Arush Arora

## Introduction

This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the shortest-distance paths of the graph before expecting it to serve the LLM with **multiplicative** GREPs for navigation tasks, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

### The R-PEARL GNN

The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

#### Graph Convolutional Network (GNN)

The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

#### Random Graph Positional Encodings (R-PEARL)

The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $$\mathbf{Q} \in \mathbb{R}^{M \times N} \qquad \mathbf{Q} \sim \mathcal{N}(0, \mathbf{I}) \qquad \mathbf{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\mathbf{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\mathbf{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\mathbf{P}$:$$\mathbf{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)};\, \mathbf{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \mathbf{S}^k\mathbf{q}^{(m)} {\mathbf{H}}_k\bigg)$$ $$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \mathbf{P}^{(m)}$$

### Sparse Graph Transformer

The Sparse Graph Transformer (hereafter named Graph Transformer or GT) follows the same architecture as that of a normal transformer, albeit that the attention mecahnism is modified to scope only over the $k$-hop neighborhood of the query node. The mathematical equations below express the functionality of the Graph Transformer:
$$\mathbf{X}_L = \Phi\Big(\mathbf{X}_0 + \mathbb{\hat{E}}_{\mathbf{q \sim \mathcal{N}(0,\, \mathbf{I})}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}\right]^\top_{\begin{subarray}{l}t \in [N] \\[2.5pt] U = \mathcal{N}^{\le k}(t)\end{subarray}}$$
$$\mathbf{Y}^{(h)}_l = \left(\mathbf{W}_o\right)^\top_l \mathbf{V}_l \mathbf{X}_{l - 1} \left(\mathbf{A}^{(h)}_l\right)^\top$$
$$\mathbf{X}_l = \sigma\bigg(\sum_{h = 1}^H \mathbf{Y}^{(h)}_l\bigg)$$

### Transformer

The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$$\mathbf{Z}_{1:t}^{(L)} = \operatorname{Trf}\Big(\mathbf{X}_{1:t}, {\mathcal{T}}_l\Big) \qquad {\mathcal{T}}_l = \begin{bmatrix}
\mathbf{Q}_l & \mathbf{K}_l & \mathbf{V}_l & \left(\mathbf{W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big(\mathbf{Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

### Graph-Augmented LLM

The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \mathbf{S}, \mathcal{H}\Big)$$

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$${\mathbf{Z}}_{1:t}^{(L)} = \operatorname{Trf}\Big({\mathbf{X}}_{1:t}; \, \cdot \,\Big)$$

## Setup

In [1]:
# %env CUDA_VISIBLE_DEVICES=0
%load_ext autoreload
%autoreload 2

In [2]:
# Import modules.
import gc
import copy
import wandb
import torch
import random
import pickle
import sympy as sp
import networkx as nx

from typing import Union

from torch import nn
from torch_geometric.data import Data
from torch.nn.utils import clip_grad_norm_
from torch_geometric.utils import to_networkx
from torch_geometric.loader import DataLoader
from torch.distributions import Cauchy, Normal
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.utils import to_dense_adj, to_networkx

from prism.models.gt import GraphTransformer, SemanticGraphTransformer
from prism.models.r_pearl import RandomGNNPositionalEncodings
from prism.models.gcn import GCN
from prism.data import data, utils

In [3]:
# Weights & Biases setup. Mirrors prism.training.train_v3._setup_wandb (project /
# name / tags / group + full-config logging), adapted for this notebook's hand-written
# train loops. Each training stage gets its own run, grouped/tagged by GNN type so the
# R-PEARL and GT variants of the same stage line up on one W&B dashboard. The helpers
# introspect the live optimizer / scheduler / loss objects so EVERY hyperparameter is
# logged without hand-maintaining a list.
WANDB_PROJECT = 'e9-gnn-navigation'


def optimizer_hparams(optimizer):
    """Every optimizer setting: class name, shared defaults, and per-param-group values
    (LRs, betas, eps, weight_decay, ...) with the parameter tensors stripped out."""
    return {
        'optimizer': type(optimizer).__name__,
        'optimizer_defaults': dict(optimizer.defaults),
        'param_groups': [
            {k: v for k, v in g.items() if k != 'params'}
            for g in optimizer.param_groups
        ],
    }


def scheduler_hparams(scheduler):
    """Every LR-scheduler setting (or {'scheduler': None} when unused)."""
    if scheduler is None:
        return {'scheduler': None}
    keys = ('mode', 'factor', 'patience', 'threshold', 'threshold_mode',
            'cooldown', 'min_lrs', 'eps')
    return {
        'scheduler': type(scheduler).__name__,
        **{k: getattr(scheduler, k) for k in keys if hasattr(scheduler, k)},
    }


def loss_hparams(loss_fn):
    """Loss class, reduction, and pos_weight (resolved to plain Python)."""
    out = {'loss_fn': type(loss_fn).__name__,
           'reduction': getattr(loss_fn, 'reduction', None)}
    pos_weight = getattr(loss_fn, 'pos_weight', None)
    if pos_weight is not None:
        out['pos_weight'] = (pos_weight.detach().cpu().tolist()
                             if torch.is_tensor(pos_weight) else pos_weight)
    return out


def init_wandb(stage, hparams):
    """Start a W&B run for a training `stage` ('edge_detection' / 'path_navigation').

    Logs the FULL run config: the GNN construction kwargs (`model_hparams`, set in the
    GNN-instantiation cell) plus every optimizer / scheduler / loss / batching
    hyperparameter the caller assembles in `hparams`. `model_type` selects R-PEARL vs
    GT and drives the run name / tag / group. Returns the run; `reinit=True` so
    successive stages in one notebook session each open a fresh run.
    """
    return wandb.init(
        project=WANDB_PROJECT,
        name=f'{stage}_{model_type}',
        tags=[stage, model_type],
        group=model_type,
        config={'model_type': model_type, 'stage': stage,
                'model': model_hparams, **hparams},
        reinit='return_previous',
    )

In [4]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 3, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [5]:
# Standard options.
ex_path = '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
eval_path = ex_path # '../data_store/old/eval/e6_transferability'
save_path = '../data/pickle/e6_eval_graphs.pkl'
device = 'cuda'

In [6]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = data.load_samples_by_graph(ex_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [random.choice(eval_data)]}

In [7]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_004.html


## Experiments

### §1 Pretraining a GNN to Classify Edge Existence

We first hope to optimize a GNN (R-PEARL or Graph Transformer) to classify whether an edge exists in the graph or not. Such a model will serve as a backbone pretrained model for fine-tuning on reporting shortest paths. The equations to represent this procedure are below:
$$\mathbf{H} = \mathbf{\Psi} = \Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{\hat{y}}_{ij} = \text{MLP}\big[\mathbf{h}_i\ \Vert\ \mathbf{h}_j\ \Vert\ \mathbf{h}_i \odot \mathbf{h}_j\ \Vert\ |\mathbf{h}_i - \mathbf{h}_j\|\big] \in [0, 1]$$

#### Model Definitions

We first define the models.

In [8]:
# Instantiate a GNN. `model_type` / `model_hparams` are exposed at module scope so
# init_wandb can log the GNN config; create_gnn writes model_hparams as it builds.
def create_gnn(model_type: str):
    global model_hparams
    if model_type == 'gt':
        model_hparams = dict(
            num_layers=3,
            pe_hidden_channels=256,
            pe_num_layers=5,
            d_model=1024,
            heads=8,
            num_samples=320,
            dropout=0.1,
            k_pe=3,
            k_gt=2,
            eps=1e-6,
            use_layer_norm=True,
        )
        gnn = GraphTransformer(**model_hparams)
        gnn.out_features = gnn.d_model
    else:
        model_hparams = dict(
            pe_hidden_channels=256,
            pe_num_layers=5,
            d_model=1024,
            num_samples=320,
            dropout=0.1,
            k=3,
            eps=1e-6,
            use_layer_norm=True,
        )
        gnn = RandomGNNPositionalEncodings(**model_hparams)
        gnn.out_features = gnn.output_projection.out_features
    return gnn


model_type = 'gt'
gnn = create_gnn(model_type)

In [9]:
# Define a class for edge detection and instantiate it.
class GNNEdgeDetector(nn.Module):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super(GNNEdgeDetector, self).__init__()
        self.gnn = gnn
        shape = self.gnn.out_features
        self.classifier = nn.Sequential(
            nn.Linear(4 * shape, shape),
            nn.LeakyReLU(),
            nn.Linear(shape, 1)
        )
        self.graph = Data(
            x=torch.empty((0, 0), dtype=torch.float),
            edge_index=torch.empty((2, 0), dtype=torch.long)
        )
        self.cached_pe = torch.zeros(size=(1, shape))

    def forward(self, graph: Data, node1: int, node2: int):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)
        hi, hj = self.cached_pe[node1], self.cached_pe[node2]
        return self.classifier(torch.cat((hi, hj, hi * hj, abs(hi - hj)), dim=0))
    
    def invalidate_cache(self):
        self.graph = None
        self.cached_pe = None


# Instantiate the class.
detector = GNNEdgeDetector(gnn)

#### Numeric Visualizations with SymPy

Using the `render_matrix()` function defined at the very beginning of this notebook, we explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the graph adjacency given a scene graph PyTorch `Data` object.

In [10]:
# Prepare a graph from the data to be used in the GNN.
load_ex_graph = False

if load_ex_graph:
    with open(save_path, 'rb') as file:
        ex_graph = pickle.load(file)[4]
        N = ex_graph.num_nodes
else:
    ex_graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
    adj = to_dense_adj(ex_graph.edge_index).squeeze().cuda()

    N = ex_graph.num_nodes
    ex_graph.edge_index = ex_graph.edge_index.to(device)
    ex_graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, 1)).to(device)

    EPS = 1e-12
    MAX_LENGTH = 128
    g = to_networkx(ex_graph, to_undirected=True, edge_attrs=['distance_m'])
    ex_graph.nxg = g
    all_pairs = dict(nx.all_pairs_dijkstra(g, weight='distance_m'))
    delta_max = max(len(path) for target in all_pairs.values() for path in target[1].values())
    paths = torch.zeros((N, N, delta_max if delta_max < MAX_LENGTH else MAX_LENGTH)).long()
    dist = torch.full((N, N), float('inf'))
    for u, (lengths_u, paths_u) in all_pairs.items():
        for v, p in paths_u.items():
            dist[u, v] = lengths_u[v]
            p = (
                torch.tensor(p, device=device).long() if len(p) < MAX_LENGTH 
                else torch.full((MAX_LENGTH,), -1, device=device).long()
            )
            paths[u, v, 0:len(p)] = p
            paths[v, u, 0:len(p)] = p
    dist.fill_diagonal_(EPS)
    ex_graph.paths = paths.to(device)
    ex_graph.dist = dist.to(device)

    # Dense adjacency (bool).
    ex_graph.adj = to_dense_adj(
        ex_graph.edge_index, max_num_nodes=N
    ).squeeze(0).bool().to(device)
    ex_graph.adj_w = to_dense_adj(
        ex_graph.edge_index, edge_attr=ex_graph.distance_m.to(device), max_num_nodes=N
    ).squeeze(0).to(device)

# Show the shortest paths matrix of a node in the graph.
node1 = random.randint(0, N - 1)
node2 = random.randint(0, N - 1)
render_matrix(ex_graph.paths[node1, node2][None, :], sig_figs=0)

Matrix([[5, 24, 20, 3, 0, 0, 0, 0, 0]])

In [11]:
# Feed the matrix to the GNN.
gnn.eval()
with torch.no_grad():
    out = gnn(ex_graph).to(device)

_, _, V = torch.pca_lowrank(out, q=10, center=True)
out = out - out.mean(dim=0)
render_matrix(out @ V)

Matrix([
[  1.82, 0.0431,   -0.205,   -0.147,   -0.323,   -0.113,  -0.00634,  -0.0327,    0.0747,    0.0235],
[  2.03,  0.158,    -0.33,    0.195,   0.0849,   0.0106,    0.0303,  -0.0874,    0.0748,   -0.0663],
[ 0.148,  0.957,    0.986,   -0.811,    0.245,  -0.0509, -0.000556,  -0.0373,   -0.0136,   -0.0425],
[ -1.41,  0.642, -0.00792,  0.00633,    -0.21,   0.0451,   0.00349,    0.026,   -0.0215,   -0.0401],
[ -1.32,  0.678,  -0.0801,     0.13,  -0.0769,   0.0872,    0.0398,  0.00213,    0.0223,   -0.0675],
[ -1.27,  0.644,   0.0176,  -0.0423,   -0.189,   0.0473,    0.0422,   0.0246,    0.0744,    0.0476],
[-0.898, -0.994,    0.179,   -0.151,   -0.263,  -0.0577,   -0.0459,   0.0647,    0.0137,    0.0338],
[0.0789, -0.772,    0.637,   -0.408,    0.157,  -0.0515,   -0.0344,   0.0123,    0.0438,   -0.0101],
[ -0.54, -0.887,   0.0267,    0.111,   0.0824,   0.0488,    0.0465,   0.0116,     0.151,   -0.0302],
[   2.0,  0.148,   -0.352,     0.15,    0.159,  -0.0495,    0.0563,   0.0938,   -0

In [12]:
# Test out the Detector.
detector.eval()
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
with torch.no_grad():
    out = detector(ex_graph, node1, node2).to(device)

print(node1, node2)
render_matrix(out.sigmoid())

10 22


Matrix([[0.443]])

#### Pre-Training of GNN on Edge Incidence

Next, we actually preprocess and train the GNN using the steps defined above.

In [13]:
# Init variables.
load_test_graphs = False

# Configure the training and test datasets.
train_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/train_graphs'
)
test_dataset, _ = data.load_samples_by_graph(
    eval_path
)

# Configure the validation dataset.
EPS = 1e-12
MAX_LENGTH = 128
train_prop = 0.8
train_num = len(train_dataset)
train_keys = random.sample(list(train_dataset.keys()), k=int(train_num * train_prop))
val_dataset = {k: v for k, v in train_dataset.items() if k not in train_keys}
train_dataset = {k: v for k, v in train_dataset.items() if k in train_keys}

# Preprocess the data.
def generate_data(dataset):
    graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in dataset.items()]
    for graph in graphs:
        N = graph.num_nodes
        graph.edge_index = graph.edge_index.to(device)
        graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, 1)).to(device)

        # Distances and paths.
        g = to_networkx(graph, to_undirected=True, edge_attrs=['distance_m'])
        graph.nxg = g
        all_pairs = dict(nx.all_pairs_dijkstra(g, weight='distance_m'))
        delta_max = max(len(path) for target in all_pairs.values() for path in target[1].values())
        paths = torch.full((N, N, delta_max if delta_max < MAX_LENGTH else MAX_LENGTH), -1).long()
        dist = torch.full((N, N), float('inf'))
        for u, (lengths_u, paths_u) in all_pairs.items():
            for v, p in paths_u.items():
                dist[u, v] = lengths_u[v]
                p = (
                    torch.tensor(p, device=device) if len(p) < MAX_LENGTH 
                    else torch.full((MAX_LENGTH,), -1, device=device)
                )
                paths[u, v, 0:len(p)] = p
                paths[v, u, 0:len(p)] = p
        dist.fill_diagonal_(EPS)
        graph.paths = paths.to(device)
        graph.dist = dist.to(device)

        # Dense adjacency (bool).
        graph.adj = to_dense_adj(
            graph.edge_index, max_num_nodes=N
        ).squeeze(0).bool().to(device)
        graph.adj_w = to_dense_adj(
            graph.edge_index, edge_attr=graph.distance_m.to(device), max_num_nodes=N
        ).squeeze(0).to(device)

        # Edges.
        combs = torch.triu_indices(N, N, offset=1, device=device)
        edge_codes = graph.edge_index[0] * N + graph.edge_index[1]
        existence = torch.isin(combs[0] * N + combs[1], edge_codes)
        graph.exclusion = combs[:, ~existence].to(device)
        indices = torch.randint(
            high=graph.exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, graph.exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones((graph.edge_index.shape[1],)), torch.zeros((indices.shape[0],))), dim=0
        ).to(device)

    return graphs


def reshuffle(graphs):
    """Function for reshuffling edge data during training."""
    for graph in graphs:
        indices = torch.randint(
            high=graph.exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, graph.exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones((graph.edge_index.shape[1],)), torch.zeros((indices.shape[0],))), dim=0
        ).to(device)

    return graphs


train_graphs = generate_data(train_dataset)
val_graphs = generate_data(val_dataset)

test_dataset = {k: v for k, v in test_dataset.items() if k not in ['eval_graph_unique_1000']}
if load_test_graphs:
    with open(save_path, 'rb') as file:
        test_graphs = pickle.load(file)
else:
    test_graphs = generate_data(test_dataset)
    with open(save_path, 'wb') as file:
        pickle.dump(test_graphs, file)

In [14]:
# Train the GNNEdgeDetector to reconstruct the graph adjacency.
batch_size = 4
val_freq = 5
epochs = 150
es_patience = 5
train_edges = False

def test_loop_edges(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss, correct = 0, 0
    tp = fp = fn = tn = 0

    with torch.no_grad():
        for idx in order.tolist():
            graph = dataloader.dataset[idx]
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            test_loss += loss_fn(preds, graph.edges_y).item()
            true = graph.edges_y.bool()
            pred = preds > 0
            tp += (pred & true).sum().item()
            fp += (pred & ~true).sum().item()
            fn += (~pred & true).sum().item()
            tn += (~pred & ~true).sum().item()
            correct += ((preds.sigmoid() > 0.5).float() == graph.edges_y).float().mean().item()

    test_loss /= size
    correct /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, F1: {f1:.3f} | P: {precision:.3f} "
          f"| R: {recall:.3f} | Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss:>8f} \n")
    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss': test_loss,
            f'{wandb_prefix}/accuracy': correct,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc,
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, f1


def train_loop_edges(train_dataloader, val_dataloader, test_dataloader, model,
               loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    val_loss: float = 0
    best_val, best_state, bad_runs = float('inf'), None, 0
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('edge_detection', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        reshuffle(val_dataloader.dataset)
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq + 1}\n=============")
            val_loss, _ = test_loop_edges(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss)
            if val_loss < best_val - 1e-3:
                best_val, bad_runs = val_loss, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val {best_val:>8f})")
                    break
            model.train()
        
        print(f"=============\nEpoch #{i + 1}\n=============")
        optimizer.zero_grad()
        order = torch.randperm(size)
        reshuffle(train_dataloader.dataset)
        for j, idx in enumerate(order.tolist()):
            # Compute prediction and loss.
            graph = train_dataloader.dataset[idx]
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            loss = loss_fn(preds, graph.edges_y)

            # Backpropagation.
            (loss / batch_size).backward()
            model.invalidate_cache()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                loss, current = loss.item(), j
                wandb.log({
                    'train/loss': loss,
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state is not None:
        model.load_state_dict(best_state)
    
    # Test the finished model.
    test_loop_edges(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


loss_fn = nn.BCEWithLogitsLoss()
train_dataloader = DataLoader(train_graphs, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs, batch_size=batch_size)
if train_edges:
    optimizer = torch.optim.AdamW([
        {'params': detector.gnn.parameters(), 'lr': 3e-5},
        {'params': detector.classifier.parameters(), 'lr': 3e-4},
    ], betas=(0.9, 0.95), weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    train_loop_edges(train_dataloader, val_dataloader, test_dataloader, detector,
                     loss_fn, optimizer, scheduler, batch_size=batch_size, epochs=epochs)
    del optimizer, scheduler
    gc.collect()

In [15]:
if train_edges:
    torch.save(detector, '../outputs/e9_multistage_training/edge_detector.pt')
    torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/edge_detector_{model_type}.pt')
else:
    detector = torch.load('../outputs/e9_multistage_training/special/edge_detector_final.pt', weights_only=False)
    gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/special/edge_detector_{model_type}_final.pt'))

#### Evaluation of Pre-Trained GNN on Edge Incidence

We now test the trained model on the evaluation dataset. First, we we will render the output for clarity.

In [16]:
# Test out the Detector.
detector.eval().to(device)
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
with torch.no_grad():
    out = detector(ex_graph, node1, node2)

print(node1, node2)
render_matrix(out.sigmoid())

16 13


Matrix([[0.162]])

In [17]:
# Evaluate the GNN on its reconstruction of test graph adjacencies.
test_loop_edges(test_dataloader, detector, loss_fn)
pass

Test Error: 
 Accuracy: 91.5%, F1: 0.920 | P: 0.857 | R: 0.993 | Bal Acc: 91.4% | Avg loss: 0.255006 



### §2 Fine-tuning the GNN to Estimate Shortest-Path Distances

We now wish to optimize the pre-trained GNN (R-PEARL or Graph Transformer) to estimate the distance of the shortest path between two given nodes in the graph. Such a model will assist the shortest-path prediction model, serving as the backbone for the multi-stage training loop featured in E9 Multistage Training of the GREP-PRISM project. The equations to represent this procedure are below:
$$\mathbf{H} = \mathbf{\Psi} = \Phi\bigg(\Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big);\, S, \mathcal{H}\bigg)$$
$$c_2(\Psi, \Psi) = \sqrt{2\operatorname{diag}(\Psi^2) - 2\Psi^2} \approx [SPD]$$
$$\mathbf{E} = \mathbb{E}\left[\frac{[SPD]_{ij}}{\delta(i, j)}\right]_{i, j \in [N]}$$

#### Model Definitions

We first define the model by attaching a simple GCN head to the GNN positional encoder.

In [18]:
# Define a class for shortest-path distance estimation and instantiate it.
class GNNShortestPathsEstimator(nn.Module):
    """
    Simple class to predict the shortest-path distance graphical lasso estimator (covariance).
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super(GNNShortestPathsEstimator, self).__init__()
        self.head = GCN(
            model_hparams['d_model'],
            model_hparams['d_model'],
            model_hparams['num_layers'],
            use_random_walk=True,
            skip_connection=True,
            dropout=model_hparams['dropout'],
            k=model_hparams['k_pe']
        )
        self.gate = nn.Parameter(torch.tensor(0.1))
        self.gnn = gnn

    def forward(self, graph: Data):
        graph = graph.clone()
        graph.x = self.gnn(graph)
        out = self.head(graph)
        out = torch.cdist(out, out, p=2)
        return self.gate * out

#### Numeric Visualizations with SymPy

Using the `render_matrix()` function defined at the very beginning of this notebook, we first explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the shortest-path distances matrix given a scene graph PyTorch `Data` object.

In [19]:
# Test out the SPD GNN.
spd_gnn = GNNShortestPathsEstimator(detector.gnn).eval()
with torch.no_grad():
    out = spd_gnn(ex_graph).to(device)

render_matrix(out)

Matrix([
[   0, 1.59,    2.73,    1.84,  1.89,   1.56,   1.8,     1.8,   1.87,    2.51,   1.57,   1.83,  1.71,  1.47,    4.72,    1.73,    2.03, 2.63,  2.16, 4.83,   2.02, 2.39,   2.03,  2.28,  2.19,  2.18,  2.49,     3.5,    2.34,   1.9,  2.54,    2.07,  2.18],
[1.59,    0,    2.44,    1.54,  1.49,   1.83,  2.47,    2.08,   2.22,    2.13,    1.5,   1.56,  1.98,  1.33,    4.87,    1.51,    1.84, 2.38,  2.17,  4.7,   1.96, 2.46,   1.97,  2.25,  2.14,  2.42,  2.77,    3.75,    2.74,  2.48,  2.67,     2.4,  2.52],
[2.73, 2.44, 0.00191,    2.15,  2.15,   2.14,  2.91,    2.34,   2.82,    3.27,   2.66,   2.72,  3.04,  2.59,    5.39,    2.63,    2.99,  1.7,  2.38, 5.07,   2.19,  2.6,   2.19,  2.04,   2.3,  2.87,  2.98,    3.84,    3.13,   2.9,  2.95,    2.76,  2.86],
[1.84, 1.54,    2.15, 0.00156, 0.218,  0.992,  2.19,    1.89,   1.87,    2.39,   1.89,   1.81,  2.03,  1.76,    4.79,    1.84,     2.1, 1.96,  1.23, 4.32,   1.17, 1.51,   1.16,  1.35,  1.26,  2.13,  2.17,    3.18,     2.1,  2.14,

In [20]:
# Render shortest-paths matrix.
render_matrix(ex_graph.dist)

Matrix([
[1.0e-12,    22.8,   184.0,   167.0,   175.0,   184.0,   140.0,   153.0,   132.0,    27.5,    16.2,    4.72,    20.3,    16.2,    40.0,    20.5,    25.7,   182.0,   157.0,   135.0,   164.0,   183.0,   172.0,   167.0,   182.0,   136.0,   144.0,   153.0,   113.0,   139.0,   148.0,   127.0,   130.0],
[   22.8, 1.0e-12,   170.0,   153.0,   161.0,   170.0,   153.0,   166.0,   145.0,    34.8,    9.41,    18.1,    29.7,    23.9,    50.0,    2.32,    18.9,   168.0,   143.0,   121.0,   150.0,   169.0,   158.0,   153.0,   168.0,   149.0,   157.0,   166.0,   127.0,   152.0,   161.0,   141.0,   143.0],
[  184.0,   170.0, 1.0e-12,    23.3,    29.8,    37.3,   115.0,   117.0,   117.0,   200.0,   175.0,   180.0,   164.0,   189.0,   215.0,   168.0,   184.0,    2.18,    27.1,    49.0,    20.1,    37.8,    27.3,    16.9,    35.6,   111.0,    95.9,   128.0,   126.0,   120.0,   112.0,   112.0,   118.0],
[  167.0,   153.0,    23.3, 1.0e-12,    18.8,    24.9,   105.0,   106.0,   106.0,   183.0,   1

In [21]:
# Render error matrix.
render_matrix(out / ex_graph.dist)

Matrix([
[      0,  0.0697,  0.0148,   0.011,  0.0108, 0.00851, 0.0129,  0.0118, 0.0142,  0.0913, 0.0971,   0.387, 0.0839,  0.0909,   0.118,  0.0845,  0.0791, 0.0144, 0.0138, 0.0357,  0.0123, 0.0131,  0.0118, 0.0136, 0.0121,  0.016, 0.0173,  0.0229,  0.0206, 0.0137, 0.0172,  0.0163, 0.0168],
[ 0.0697,       0,  0.0144,  0.0101, 0.00927,  0.0108, 0.0161,  0.0125, 0.0153,  0.0612,   0.16,   0.086, 0.0665,  0.0558,  0.0973,   0.652,  0.0976, 0.0141, 0.0151, 0.0388,   0.013, 0.0146,  0.0125, 0.0147, 0.0128, 0.0162, 0.0177,  0.0226,  0.0216, 0.0163, 0.0166,   0.017, 0.0176],
[ 0.0148,  0.0144, 1.91e+9,  0.0924,  0.0722,  0.0574, 0.0253,    0.02, 0.0241,  0.0163, 0.0152,  0.0151, 0.0186,  0.0137,  0.0251,  0.0157,  0.0162,   0.78, 0.0876,  0.103,   0.109, 0.0689,  0.0802,  0.121, 0.0646, 0.0257,  0.031,    0.03,  0.0249, 0.0242, 0.0263,  0.0246, 0.0243],
[  0.011,  0.0101,  0.0924, 1.56e+9,  0.0117,  0.0398, 0.0209,  0.0178, 0.0176,   0.013,  0.012,  0.0111, 0.0138,  0.0102,  0.0242,  0.0122

#### Definition of a Custom Loss Function: Graphical Lasso Estimator 

We seek to reproduce the [Graphical Lasso Estimator](https://en.wikipedia.org/wiki/Graphical_lasso) custom loss function within the PyTorch framework. Since such an error and gradient computation function requires a differentiable interpretation of the $L_1$ regularization penalty, we must define a new subclass of `torch.autograd.Function` to implement this regression objective within the working environment.

The Graphical Lasso Estimator is defined through the following mathematical optimizer:

$$\hat{\Theta} = \argmax_{\Theta \succ 0} L(\Theta) = \argmax_{\Theta \succ 0}\left(\log\det(\Theta) - \operatorname{tr}(S\Theta) - \lambda\sum_{i, j}|\Theta_{ij}|\right)$$

Thus, it has the following derivative evaluation:

$$\nabla_{\Theta} L(\Theta) = \frac{1}{\det(\Theta)} \det(\Theta) \Theta^{-\top} - S^T - \lambda \begin{cases}1 & \text{if } \Theta_{ij} > 0 \\ 0 & \text{if } \Theta_{ij} = 0 \\ -1 & \text{if } \Theta_{ij} < 0\end{cases}$$
$$\nabla_{\Theta} L(\Theta) = \Theta^{-1} - S - \lambda \operatorname{sign}(\Theta)$$

In [22]:
LAMBDA = 1e-7


class GraphicalLassoEstimator(torch.autograd.Function):
    @staticmethod
    def forward(ctx, preds, targets):
        """
        Computes the loss value for the Graphical Lasso Estimator loss function.
        """
        ctx.save_for_backward(preds, targets)
        sign, logdet = torch.linalg.slogdet(preds)
        assert (sign > 0).all()
        return -logdet + torch.trace(targets @ preds) - LAMBDA * preds.abs().sum()

    @staticmethod
    def backward(ctx, grad_output):
        """
        Computes custom gradients with respect to the inputs. Honors the requirement
        for L1 differentiability within the PyTorch framework.
        """
        preds, targets = ctx.saved_tensors
        grad_predictions = grad_targets = None
        if ctx.needs_input_grad[0]:
            grad_predictions = grad_output * - (preds.inverse() - targets - LAMBDA * preds.sign())
        if ctx.needs_input_grad[1]:
            grad_targets = grad_output * preds
        return grad_predictions, grad_targets

#### Fine-Tuning of GNN on Shortest-Path Distances

We preprocess and train the GNN using the steps defined above.

In [23]:
# Train the GNN to reconstruct the shortest-path distances of the graph.
batch_size = 4
val_freq = 5
epochs = 200
es_patience = 5
train_dists = False

def test_loop_dists(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model['dists'].to(device).eval()
    model['edges'].to(device).eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss = {'dists': 0, 'edges': 0}
    correct, error_norm = 0, 0
    tp = fp = fn = tn = 0
    
    with torch.no_grad():
        for idx in order.tolist():
            graph = dataloader.dataset[idx]
            preds = {
                'dists': model['dists'](graph),
                'edges': torch.stack([
                    model['edges'](graph, graph.edges_x[0, k], graph.edges_x[1, k])
                    for k in range(graph.edges_x.shape[1])
                ]).squeeze(-1).to(device)
            }

            test_loss['dists'] += loss_fn['dists'](preds['dists'], graph.dist).item()
            error = preds['dists'] / graph.dist
            error.fill_diagonal_(0)
            error_norm += torch.linalg.matrix_norm(error) / error.shape[0]

            test_loss['edges'] += loss_fn['edges'](preds['edges'], graph.edges_y).item()
            true = graph.edges_y.bool()
            pred = preds['edges'] > 0
            tp += (pred & true).sum().item()
            fp += (pred & ~true).sum().item()
            fn += (~pred & true).sum().item()
            tn += (~pred & ~true).sum().item()
            correct += ((preds['edges'].sigmoid() > 0.5).float() == graph.edges_y).float().mean().item()

    test_loss['dists'] /= size
    error_norm /= size
    print(f"Test Error #1: \n Avg error: {error_norm:>0.3f} \n Avg loss: {test_loss['dists']:>8f} \n")

    test_loss['edges'] /= size
    correct /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    print(f"Test Error #2: \n Accuracy: {(100*correct):>0.1f}%, F1: {f1:.3f} | P: {precision:.3f} "
          f"| R: {recall:.3f} | Bal Acc: {(100*bal_acc):.1f}% | Avg loss: {test_loss['edges']:>8f} \n")

    # Log eval metrics under the given split prefix (e.g. 'val') when a run is active.
    if wandb_prefix is not None and wandb.run is not None:
        log = {
            f'{wandb_prefix}/loss_dists': test_loss['dists'],
            f'{wandb_prefix}/error_norm': error_norm,
            f'{wandb_prefix}/loss_edges': test_loss['edges'],
            f'{wandb_prefix}/accuracy': correct,
            f'{wandb_prefix}/f1': f1,
            f'{wandb_prefix}/precision': precision,
            f'{wandb_prefix}/recall': recall,
            f'{wandb_prefix}/bal_acc': bal_acc
        }
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        # Final test metrics: also surface as run-summary headline numbers.
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return test_loss, error_norm


def train_loop_dists(train_dataloader, val_dataloader, test_dataloader, model,
                     loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model['dists'].to(device).train()
    model['edges'].to(device).train()
    val_loss: float = 0
    best_val, best_state, bad_runs, best_state = float('inf'), None, 0, {}
    # Log the full run config: GNN (model_hparams) + optimizer/scheduler/loss/batching.
    run = init_wandb('shortest_path_distances', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn['dists']),
        **loss_hparams(loss_fn['edges']),
    })
    global_step = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq + 1}\n=============")
            val_loss, _ = test_loop_dists(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(val_loss['dists'])
            if (val_loss['dists'] < best_val['edges'] - 1e-3 
                    and val_loss['edges'] < best_val['edges'] - 1e-3):
                best_val, bad_runs = val_loss, 0
                best_state['dists'] = copy.deepcopy(model['dists'].state_dict())
                best_state['edges'] = copy.deepcopy(model['edges'].state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best val {best_val:>8f})")
                    break
            model['dists'].train()
            model['edges'].train()
        
        print(f"=============\nEpoch #{i + 1}\n=============")
        optimizer.zero_grad()
        order = torch.randperm(size)
        reshuffle(train_dataloader.dataset)
        for j, idx in enumerate(order.tolist()):
            # Compute prediction and loss.
            graph = train_dataloader.dataset[idx]
            preds = {
                'dists': model['dists'](graph),
                'edges': torch.stack([
                    model['edges'](graph, graph.edges_x[0, k], graph.edges_x[1, k])
                    for k in range(graph.edges_x.shape[1])
                ]).squeeze(-1).to(device)
            }
            preds['dists'] = preds['dists']
            loss = {
                'dists': loss_fn['dists'](preds['dists'], graph.dist),
                'edges': loss_fn['edges'](preds['edges'], graph.edges_y)
            }

            # Backpropagation.
            loss['dists'] /= graph.num_nodes
            ((loss['dists'] + loss['edges']) / batch_size).backward()
            model['edges'].invalidate_cache()

            # Optimization and results.
            if (j + 1) % batch_size == 0:
                clip_grad_norm_(model['dists'].parameters(), max_norm=1.0)
                clip_grad_norm_(model['edges'].parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                current = j
                wandb.log({
                    'train/loss_dists': loss['dists'],
                    'train/loss_edges': loss['edges'],
                    'train/lr': optimizer.param_groups[0]['lr'],
                    'epoch': i,
                    'global_step': global_step,
                })
                print(f"Loss #1: {loss['dists'].item():>7f}  [{current:>5d}/{size:>5d}]")
                print(f"Loss #2: {loss['edges'].item():>7f}  [{current:>5d}/{size:>5d}]")
        
    # Early stopping hatch.
    if best_state:
        model['dists'].load_state_dict(best_state['dists'])
        model['edges'].load_state_dict(best_state['edges'])

    # Test the finished model.
    test_loop_dists(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


# Establish MSE/Graphical-Lasso loss.
loss_fn = {
    'dists': nn.MSELoss(),
    'edges': nn.BCEWithLogitsLoss()
}
if train_dists:
    optimizer = torch.optim.AdamW([
        {'params': gnn.parameters(), 'lr': 3e-5},
        {'params': detector.classifier.parameters(), 'lr': 3e-4},
        {'params': spd_gnn.head.parameters(), 'lr': 3e-4},
        {'params': [spd_gnn.gate], 'lr': 3e-4}
    ], betas=(0.9, 0.95), weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    train_loop_dists(train_dataloader, val_dataloader, test_dataloader, 
                    {'dists': spd_gnn, 'edges': detector}, loss_fn, optimizer, 
                    scheduler, batch_size=batch_size, epochs=epochs)
    del optimizer, scheduler
    gc.collect()

In [24]:
if train_dists:
    torch.save(detector, '../outputs/e9_multistage_training/edge_detector.pt')
    torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/edge_detector_{model_type}.pt')
else:
    detector = torch.load('../outputs/e9_multistage_training/suite1/edge_detector.pt', weights_only=False)
    gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/suite1/edge_detector_{model_type}.pt'))

In [25]:
if train_dists:
    torch.save(spd_gnn, '../outputs/e9_multistage_training/spd_gnn.pt')
    torch.save(spd_gnn.gnn.state_dict(), f'../outputs/e9_multistage_training/spd_gnn_{model_type}.pt')
else:
    spd_gnn = torch.load(f'../outputs/e9_multistage_training/suite1/spd_gnn.pt', weights_only=False)
    gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/suite1/spd_gnn_{model_type}.pt'))

#### Evaluation of Pre-Trained GNN on Edge Incidence and Shortest-Paths Distance Estimation

We test the pre-trained model on the evaluation dataset. We we will render the output error matrix $\mathbf{E}$ for visibility.

In [26]:
# Test out the SPD GNN.
spd_gnn.eval().to(device)
with torch.no_grad():
    out = spd_gnn(ex_graph)

render_matrix(out)

Matrix([
[    0,  9.84, 180.0, 159.0, 157.0, 164.0, 155.0, 185.0,  161.0,   9.23,   35.6,   34.8,  12.6,  27.8, 106.0,  31.6,   15.9, 182.0, 147.0,  182.0,  148.0, 149.0, 150.0, 150.0, 148.0, 146.0, 147.0, 166.0, 151.0,  145.0,  166.0, 148.0, 145.0],
[ 9.84,     0, 173.0, 152.0, 150.0, 156.0, 149.0, 179.0,  154.0,   16.5,   42.5,   41.9,  21.5,  36.1,  99.4,  39.7,   24.4, 175.0, 139.0,  176.0,  141.0, 141.0, 142.0, 142.0, 140.0, 140.0, 141.0, 159.0, 145.0,  138.0,  160.0, 142.0, 139.0],
[180.0, 173.0,     0,  52.1,  54.3,  47.9,  66.9,  70.2,   67.0,  188.0,  205.0,  207.0, 192.0, 202.0,  77.7, 206.0,  195.0,  24.5,  56.9,   42.9,   62.5,  53.9,  61.6,  55.8,  60.4,  75.1,  74.0,  70.6,  65.9,   70.9,   64.4,  74.1,  79.7],
[159.0, 152.0,  52.1, 0.027,  3.22,  10.2,  40.0,  51.7,   40.4,  167.0,  185.0,  186.0, 171.0, 182.0,  63.4, 186.0,  174.0,  35.6,  17.4,   43.5,   23.2,  12.6,  21.7,  19.7,  20.4,  48.6,  50.9,  41.6,  37.3,   44.0,   39.4,  50.1,  55.1],
[157.0, 150.0,  54.3,  

In [27]:
# Render shortest-paths matrix.
render_matrix(ex_graph.dist)

Matrix([
[1.0e-12,    22.8,   184.0,   167.0,   175.0,   184.0,   140.0,   153.0,   132.0,    27.5,    16.2,    4.72,    20.3,    16.2,    40.0,    20.5,    25.7,   182.0,   157.0,   135.0,   164.0,   183.0,   172.0,   167.0,   182.0,   136.0,   144.0,   153.0,   113.0,   139.0,   148.0,   127.0,   130.0],
[   22.8, 1.0e-12,   170.0,   153.0,   161.0,   170.0,   153.0,   166.0,   145.0,    34.8,    9.41,    18.1,    29.7,    23.9,    50.0,    2.32,    18.9,   168.0,   143.0,   121.0,   150.0,   169.0,   158.0,   153.0,   168.0,   149.0,   157.0,   166.0,   127.0,   152.0,   161.0,   141.0,   143.0],
[  184.0,   170.0, 1.0e-12,    23.3,    29.8,    37.3,   115.0,   117.0,   117.0,   200.0,   175.0,   180.0,   164.0,   189.0,   215.0,   168.0,   184.0,    2.18,    27.1,    49.0,    20.1,    37.8,    27.3,    16.9,    35.6,   111.0,    95.9,   128.0,   126.0,   120.0,   112.0,   112.0,   118.0],
[  167.0,   153.0,    23.3, 1.0e-12,    18.8,    24.9,   105.0,   106.0,   106.0,   183.0,   1

In [28]:
# Render error matrix.
render_matrix(out / ex_graph.dist)

Matrix([
[    0, 0.431, 0.977,   0.953, 0.899, 0.891,  1.11,  1.21,     1.22,    0.336,     2.19,     7.37, 0.618,  1.72,  2.66,  1.54,     0.62,   1.0,   0.938,     1.35,    0.904, 0.811,   0.869, 0.896,   0.814,  1.07,    1.03,  1.09,  1.33,     1.04,     1.13,  1.16,  1.12],
[0.431,     0,  1.01,   0.989, 0.929,  0.92, 0.971,  1.08,     1.06,    0.474,     4.52,     2.32, 0.724,  1.51,  1.99,  17.1,     1.29,  1.04,   0.974,     1.45,    0.937, 0.833,   0.898, 0.928,   0.835, 0.934,     0.9, 0.958,  1.14,    0.909,     0.99,  1.01, 0.974],
[0.977,  1.01,     0,    2.24,  1.82,  1.28, 0.581,   0.6,    0.574,     0.94,     1.17,     1.15,  1.17,  1.07, 0.362,  1.22,     1.06,  11.2,     2.1,    0.877,     3.11,  1.43,    2.26,  3.31,     1.7, 0.674,   0.771, 0.551, 0.524,    0.593,    0.575,  0.66, 0.677],
[0.953, 0.989,  2.24, 2.7e+10, 0.172, 0.408, 0.382, 0.486,    0.379,    0.912,     1.17,     1.14,  1.16,  1.06,  0.32,  1.23,     1.04,  1.69,     1.7,     1.35,     7.21,  0.46,  

In [29]:
def are_models_equal(model1, model2):
    # 1. Check if both models have the exact same state_dict keys
    if model1.state_dict().keys() != model2.state_dict().keys():
        return False
    
    # 2. Check if all parameters and buffers are exactly equal
    for key, value1 in model1.state_dict().items():
        value2 = model2.state_dict()[key]
        
        # Use torch.equal for strict element-wise and structural equality
        if not torch.equal(value1, value2):
            return False
            
    return True

are_models_equal(detector.gnn, spd_gnn.gnn)

True

In [30]:
# Evaluate the GNN on its reconstruction of test graph edge incidences and shortest-path distances together.
models = {'dists': spd_gnn, 'edges': detector}
test_loop_dists(test_dataloader, models, loss_fn)
pass

Test Error #1: 
 Avg error: 2.723 
 Avg loss: 1298.807162 

Test Error #2: 
 Accuracy: 94.1%, F1: 0.943 | P: 0.901 | R: 0.989 | Bal Acc: 94.0% | Avg loss: 0.211519 



### §3 Fine-tuning the GNN to Predict Shortest-Path Subgraph Adjacencies

We now wish to optimize the jointly fine-tuned GNN (R-PEARL or Graph Transformer) to predict the shortest path itself between two given nodes in the graph. Such a model will serve as the actual backbone for the multi-stage training loop featured in E9 Multistage Training of the GREP-PRISM project. The equations to represent this procedure are below:
$$\mathbf{H} = \mathbf{\Psi} = \Phi\Big(\mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H}\,)\big];\, \mathcal{T}\Big)$$
$$\mathbf{X} = \left[\mathbf{x}_i \sim \text{Cauchy}(0,\, \mathbf{I})\right]_{i \in [N]}^\top \in \mathbb{R}^{N \times D} \qquad D \gg N$$
$$\forall\, t_i \in T \qquad \mathbf{x}_i \sim \text{Cauchy}(i,\, \mathbf{I}) \implies \mathbf{X}(T) = \left[\mathbf{x}_t \sim \text{Cauchy}(t,\, \mathbf{I})\right]^\top_{t \in T}$$
$$\forall\, u, v \in V^2 \quad u \rightsquigarrow v \qquad \mathbf{x}_u \sim \text{Cauchy}(1,\, \mathbf{I}) \qquad \mathbf{x}_v \sim \text{Cauchy}\big(\delta(u, v),\, \mathbf{I}\big)$$
$$\qquad \hat{U}_{1} = u \in V \qquad \hat{U}_{t+1} = \Phi\Big(\mathbf{X}\big(U_{1:t}\big) + \mathbf{\Psi};\, \mathcal{T}\Big) \in V^{t + 1} \qquad \hat{U}_{1:T} = (u,\, \cdots, v) = \hat{U}(u, v) \in V^T$$
$$\mathbf{E} = \mathbb{E}\left[\frac{|\hat{U}(u, v)|}{\delta(u, v)}\right]_{u, v \in V^2}

#### Model Definitions
We first define the model by attaching a full Autoregressive Graph Transformer (AGT) to the GNN positional encoder.

In [31]:
# Define a class for edge detection and instantiate it.
class GNNShortestPathNavigator(GNNEdgeDetector):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer], max_length=128):
        super().__init__(gnn)
        self.MAX_LENGTH = max_length
        self.shape = gnn.out_features
        self.head = SemanticGraphTransformer(
            node_feature_dim=model_hparams['d_model'],
            num_layers=model_hparams['num_layers'],
            d_model=model_hparams['d_model'],
            heads=model_hparams['heads'],
            dropout=model_hparams['dropout'],
            k_gt=model_hparams['k_gt'],
        )
        self.classifier = nn.Linear(in_features=self.shape, out_features=1)
        self.graph = Data(
            x=torch.empty((0, 0), dtype=torch.float),
            edge_index=torch.empty((2, 0), dtype=torch.long)
        )
        self.cached_pe = torch.zeros(size=(1, self.shape))

    def forward(self, graph: Data):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)

        feed_graph = self.graph.clone()
        feed_graph.x = graph.x + self.cached_pe
        return self.classifier(self.head(feed_graph))

    def generate(self, graph: Data, node1: int, node2: int):
        """Autoregressively generates a simple path node1 -> node2 under legal-move masking."""
        # Establish Cauchy distribution.
        N, D = graph.num_nodes, self.shape
        graph.x = Normal(loc=0.0, scale=1.0).sample((N, D)).to(device)
        graph.x[node2] = Normal(loc=-5.0, scale=1.0).sample((1, D)).to(device)

        # Set up variables.
        count = 1.0
        preds = [node1]
        visited = {node1}

        # Run generation loop with adjacency + visited masking.
        while not (preds[-1] == node2 or len(preds) > self.MAX_LENGTH):
            c = preds[-1]
            graph.x[c] = Normal(loc=count, scale=1.0).sample((1, D)).to(device)
            allowed = graph.adj[c].clone()
            if visited:
                allowed[torch.as_tensor(sorted(visited), device=allowed.device)] = False
            if not bool(allowed.any()):
                break
            logits = self(graph).T.masked_fill(~allowed.unsqueeze(0), float('-inf'))
            nxt = int(logits.argmax(dim=1))
            preds.append(nxt)
            visited.add(nxt)
            count += 1.0

        # Clean up and return.
        graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, 1)).to(device)
        return torch.tensor([preds], device=device).T

    def invalidate_cache(self):
        self.graph = None
        self.cached_pe = None


# Instantiate the class.
navigator = GNNShortestPathNavigator(spd_gnn.gnn)

In [32]:
# Test out the Navigator.
navigator.eval().to(device)
with torch.no_grad():
    out = navigator(ex_graph)

render_matrix(out.softmax(dim=0).T)

Matrix([[0.0219, 0.0244, 0.0136, 0.0194, 0.0209, 0.0223, 0.0333, 0.0236, 0.0293, 0.0218, 0.0289, 0.0279, 0.0177, 0.0236, 0.0328, 0.0231, 0.0234, 0.0257, 0.0315, 0.0342, 0.0355, 0.0262, 0.0298, 0.0237, 0.046, 0.0612, 0.0213, 0.0382, 0.0211, 0.0468, 0.0328, 0.0584, 0.0598]])

In [33]:
# Test out the Navigator's generation abilities.
N = ex_graph.num_nodes
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
with torch.no_grad():
    out = navigator.generate(ex_graph, node1, node2)

print(node1, node2)
print(ex_graph.paths[node1, node2].tolist())
render_matrix(out.T, sig_figs=0)

9 16
[16, 9, 0, 0, 0, 0, 0, 0, 0]


Matrix([[9, 10, 11, 28, 32, 31, 25, 27, 19, 12, 16]])

#### Fine-Tuning of GNN on Shortest Paths
Finally, we preprocess and train the GNN using the steps defined above.

In [ ]:
# Train the GNN to predict shortest-path subgraph adjacencies (autoregressive navigator).
nav_cfg = {
    'batch_size': 4,
    'val_freq': 5,
    'epochs': 200,
    'es_patience': 5,
    'batch_prop': 1.0,
    'detour_bce': False,
    'freeze_trunk': False,
    'deterministic_pe': False,
    'eos_supervision': True,
    'eos_weight': 0.5,
    'optimal_hop_set': True,
    'opt_tol': 1e-4,
}

# Mirror the loop-control switches to module scope (closures/other cells read these names).
batch_size = nav_cfg['batch_size']
val_freq = nav_cfg['val_freq']
epochs = nav_cfg['epochs']
es_patience = nav_cfg['es_patience']
batch_prop = nav_cfg['batch_prop']
detour_bce = nav_cfg['detour_bce']
train_paths = True


# ---------------------------------------------------------------------------------------
# Decode / metric / logging helpers (shared by the train and eval loops below).
# ---------------------------------------------------------------------------------------
def _allowed_mask(adj_row, visited):
    """Legal-move mask for the current node: neighbours minus already-visited (Tasks 1-2)."""
    allowed = adj_row.clone()
    if visited:
        allowed[torch.as_tensor(sorted(visited), device=allowed.device)] = False
    return allowed


def _optimal_next_hops(graph, c, v, tol):
    """Set of neighbors w of c that lie on a shortest c->v path:
    dist[w, v] == dist[c, v] - edge(c, w)  (weighted distance_m). Task 7 optional target."""
    neigh = graph.adj[c].nonzero(as_tuple=False).view(-1).tolist()
    dcv = float(graph.dist[c, v])
    hops = [w for w in neigh
            if abs(float(graph.dist[w, v]) - (dcv - float(graph.adj_w[c, w]))) <= tol]
    return hops


def _grad_norm(params):
    """(L2 grad norm, parameter count) over params that carry a gradient."""
    grads = [p.grad.detach() for p in params if p.grad is not None]
    if not grads:
        return 0.0, 0
    total = torch.norm(torch.stack([g.norm(2) for g in grads]), 2).item()
    n = sum(g.numel() for g in grads)
    return total, n


def _grad_norm_log(modules, prefix='train'):
    """Per-module raw L2 grad norm AND size-normalised RMS (norm / sqrt(#params)) so modules
    with very different parameter counts / LRs are comparable on one axis."""
    log = {}
    for name, module in modules.items():
        total, n = _grad_norm(list(module.parameters()))
        log[f'{prefix}/grad_norm/{name}'] = total
        log[f'{prefix}/grad_rms/{name}'] = (total / (n ** 0.5)) if n else 0.0
    return log


def _set_pe_determinism(model, on):
    """Toggle R-PEARL fixed-seed probing on the shared trunk; return the prior value.
    Invalidates the navigator/detector PE caches so Psi is re-sampled under the new regime."""
    gnn = model['paths'].gnn
    prior = getattr(gnn, 'fixed_seed_mode', None)
    if prior is not None:
        gnn.fixed_seed_mode = on
        model['paths'].invalidate_cache()
        model['edges'].invalidate_cache()
    return prior


def _nav_metrics(graph, u, v, path):
    """NetworkX validation of one rollout (mirrors path_validator.validate_path):
    walk path hop-by-hop (G.has_edge + summed distance_m) vs the live
    nx.shortest_path_length optimum. Cached graph.dist/paths are not trusted here.
    """
    G = getattr(graph, 'nxg', None)
    if G is None:
        G = to_networkx(graph, to_undirected=True, edge_attrs=['distance_m'])
    node_set = set(G.nodes)

    # Walk the route hop-by-hop: node existence, edge validity, traversed cost.
    exists = [n in node_set for n in path]
    pairs = list(zip(path[:-1], path[1:]))
    edge_ok, gen_dist = [], 0.0
    for a, b in pairs:
        ok = G.has_edge(a, b)
        edge_ok.append(ok)
        if ok:
            gen_dist += G[a][b]['distance_m']
    nodes_exist = sum(exists) / len(exists)
    validity = (sum(edge_ok) / len(edge_ok)) if edge_ok else 1.0
    assert validity == 1.0, f"masked rollout emitted a non-edge {u}->{v}: {path}"
    full_valid = bool(all(exists) and all(edge_ok))
    start_goal_ok = float(path[0] == u and path[-1] == v)
    reached = float(full_valid and path[-1] == v)
    gen_hops = len(pairs)

    # Optimum run live on the graph: weighted (distance_m) and unweighted (hop BFS).
    try:
        opt_dist = nx.shortest_path_length(G, u, v, weight='distance_m')
        opt_hops = nx.shortest_path_length(G, u, v)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        opt_dist = opt_hops = float('nan')

    ratio_dist = (gen_dist / opt_dist) if (reached and opt_dist > 0) else float('nan')
    ratio_hops = (gen_hops / opt_hops) if (reached and opt_hops > 0) else float('nan')
    exact_dist = float(reached and abs(gen_dist - opt_dist) <= nav_cfg['opt_tol'])
    exact_hops = float(reached and gen_hops == opt_hops)
    spl = reached * opt_dist / max(gen_dist, opt_dist) if (
        reached and max(gen_dist, opt_dist) > 0
    ) else 0.0
    return {'success': reached, 'validity': validity, 'nodes_exist': nodes_exist,
            'start_goal_ok': start_goal_ok, 'full_valid': float(full_valid), 'spl': spl,
            'ratio_hops': ratio_hops, 'ratio_dist': ratio_dist,
            'exact_hops': exact_hops, 'exact_dist': exact_dist}


def _sample_uv(N):
    """Disjoint (u, v) endpoint pairs from a random node subset (unchanged batching scheme)."""
    batch_len = int(batch_prop * N)
    batch = torch.randperm(N)[:batch_len]
    u = batch[:batch_len // 2].tolist()
    w = batch[batch_len // 2:].tolist()
    return list(zip(u, w))


def _nanmean(xs):
    xs = [x for x in xs if x == x]
    return sum(xs) / len(xs) if xs else float('nan')


# ---------------------------------------------------------------------------------------
# Evaluation loop: free-running rollouts (masked) + NetworkX navigation metrics (Task 7),
# with a masked-CE proxy and the frozen detector's edge metrics for monitoring.
# ---------------------------------------------------------------------------------------
def test_loop_paths(dataloader, model, loss_fn, wandb_prefix=None, epoch=None):
    model['paths'].to(device).eval()
    model['edges'].to(device).eval()
    size = len(dataloader.dataset)
    order = torch.randperm(size)
    test_loss = {'paths': 0.0, 'edges': 0.0}
    correct = {'edges': 0.0}
    tp = fp = fn = tn = 0
    nav = {k: [] for k in ('success', 'validity', 'nodes_exist', 'start_goal_ok',
                           'full_valid', 'spl', 'ratio_hops', 'ratio_dist',
                           'exact_hops', 'exact_dist')}

    prior_seed = (_set_pe_determinism(model, nav_cfg['deterministic_pe']) 
                  if nav_cfg['deterministic_pe'] else None)
    with torch.no_grad():
        for idx in order.tolist():
            graph = dataloader.dataset[idx]

            # Edges.
            preds_edges = torch.stack([
                model['edges'](graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            test_loss['edges'] += loss_fn['edges'](preds_edges, graph.edges_y).item()
            true = graph.edges_y.bool()
            pred = preds_edges > 0
            tp += (pred & true).sum().item()
            fp += (pred & ~true).sum().item()
            fn += (~pred & true).sum().item()
            tn += (~pred & ~true).sum().item()
            correct['edges'] += (
                (preds_edges.sigmoid() > 0.5).float() == graph.edges_y
            ).float().mean().item()

            # Paths.
            N, D = graph.num_nodes, model['paths'].gnn.out_features
            graph_ce, n_uv = 0.0, 0
            for u, v in _sample_uv(N):
                gt = graph.paths[u, v]
                gt = gt[gt >= 0]
                if gt.numel() < 2:
                    continue
                target = gt[1:]

                count = 1.0
                path = [u]
                visited = {u}
                out_raw = []
                graph.x = Normal(loc=0.0, scale=1.0).sample((N, D)).to(device)
                graph.x[v] = Normal(loc=-5.0, scale=1.0).sample((1, D)).to(device)
                while not (path[-1] == v or len(path) > model['paths'].MAX_LENGTH):
                    c = path[-1]
                    graph.x[c] = Normal(loc=count, scale=1.0).sample((1, D)).to(device)
                    allowed = _allowed_mask(graph.adj[c], visited)
                    if not bool(allowed.any()):
                        break
                    logits = model['paths'](graph).T
                    out_raw.append(logits)
                    masked = logits.masked_fill(~allowed.unsqueeze(0), float('-inf'))
                    nxt = int(masked.argmax(dim=1))
                    path.append(nxt)
                    visited.add(nxt)
                    count += 1.0

                # Masked-Cross-Entropy proxy over the aligned overlap.
                if out_raw:
                    T = min(len(out_raw), target.shape[0])
                    if T > 0:
                        raw = torch.cat(out_raw[:T]).to(device)
                        tgt = target[:T]
                        legal = raw.new_ones(T, dtype=torch.bool)
                        for t in range(T):
                            legal[t] = bool(graph.adj[path[t], tgt[t]])
                        if legal.any():
                            graph_ce += loss_fn['paths'](raw[legal], tgt[legal]).item()
                            n_uv += 1

                for k, val in _nav_metrics(graph, u, v, path).items():
                    nav[k].append(val)

            test_loss['paths'] += graph_ce / max(n_uv, 1)
            graph.x = Cauchy(loc=0.0, scale=1.0).sample((N, 1)).to(device)

    if prior_seed is not None:
        _set_pe_determinism(model, prior_seed)

    # Aggregate.
    test_loss['paths'] /= size
    test_loss['edges'] /= size
    correct['edges'] /= size
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    bal_acc = 0.5 * (recall + tn / (tn + fp + 1e-9))
    metrics = {
        'loss_paths': test_loss['paths'],
        'loss_edges': test_loss['edges'],
        'success': _nanmean(nav['success']),
        'validity': _nanmean(nav['validity']),
        'nodes_exist': _nanmean(nav['nodes_exist']),
        'start_goal_ok': _nanmean(nav['start_goal_ok']),
        'full_valid': _nanmean(nav['full_valid']),
        'spl': _nanmean(nav['spl']),
        'opt_ratio_hops': _nanmean(nav['ratio_hops']),
        'opt_ratio_dist': _nanmean(nav['ratio_dist']),
        'exact_frac_hops': _nanmean(nav['exact_hops']),
        'exact_frac_dist': _nanmean(nav['exact_dist']),
        'accuracy': correct['edges'], 'f1': f1, 'precision': precision,
        'recall': recall, 'bal_acc': bal_acc,
    }

    print(f"Test Error #1: \n Success: {100*metrics['success']:>4.1f}% | SPL: {metrics['spl']:>.3f} | "
          f"Valid: {100*metrics['validity']:>5.1f}% \n "
          f"Reached goal: {100*metrics['start_goal_ok']:>4.1f}% | "
          f"Optimal (hops): {metrics['opt_ratio_hops']:>.3f} | "
          f"Exact (hops): {100*metrics['exact_frac_hops']:>4.1f}% \n "
          f"Avg loss: {metrics['loss_paths']:>.4f} \n")
    print(f"Test Error #2: \n Accuracy: {100*metrics['accuracy']:>0.1f}% | "
          f"F1: {f1:.3f} | P: {precision:.3f} | R: {recall:.3f} | Bal Acc: {100*bal_acc:.1f}% "
          f"| Avg loss: {metrics['loss_edges']:>.6f}\n")

    # Thorough W&B logging under the split prefix (mirrors the added lines in test_loop_dists).
    if wandb_prefix is not None and wandb.run is not None:
        log = {f'{wandb_prefix}/{k}': v for k, v in metrics.items()}
        if epoch is not None:
            log['epoch'] = epoch
        wandb.log(log)
        if wandb_prefix == 'test':
            wandb.run.summary.update({k: v for k, v in log.items() if k != 'epoch'})
    return metrics


# ---------------------------------------------------------------------------------------
# Training loop: teacher-forced, masked-CE rollouts (alignment preserved -- see Task 3 test).
# ---------------------------------------------------------------------------------------
def train_loop_paths(train_dataloader, val_dataloader, test_dataloader, model,
                     loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model['paths'].train().to(device)
    model['edges'].train().to(device)
    best_state, bad_runs = {}, 0
    best = {'spl': -1.0, 'success': -1.0}
    run = init_wandb('path_navigation', {
        'batch_size': batch_size, 'epochs': epochs,
        'val_freq': val_freq, 'es_patience': es_patience,
        'nav_cfg': nav_cfg,
        **optimizer_hparams(optimizer),
        **scheduler_hparams(scheduler),
        **loss_hparams(loss_fn['paths']),
        **loss_hparams(loss_fn['edges']),
    })

    # Modules whose gradients are optimized (and clipped / logged) this stage.
    trained = {'head': model['paths'].head, 'classifier': model['paths'].classifier}
    if not nav_cfg['freeze_trunk']:
        trained['gnn'] = model['paths'].gnn
        trained['edges'] = model['edges']

    global_step = 0
    for i in range(epochs):
        # -- Validation --
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq + 1}\n=============")
            val = test_loop_paths(val_dataloader, model, loss_fn, wandb_prefix='val', epoch=i)
            if scheduler:
                scheduler.step(1.0 - val['spl'])
            if val['spl'] > best['spl'] + 1e-3:
                bad_runs = 0
                best = {'spl': val['spl'], 'success': val['success']}
                best_state['paths'] = copy.deepcopy(model['paths'].state_dict())
                best_state['edges'] = copy.deepcopy(model['edges'].state_dict())
            else:
                bad_runs += 1
                if bad_runs >= es_patience:
                    print(f"Early stop at epoch {i} (best SPL {best['spl']:>.4f}, "
                          f"success {100*best['success']:>.1f}%)")
                    break
            model['paths'].train()
            model['edges'].train()

        print(f"=============\nEpoch #{i + 1}\n=============")
        optimizer.zero_grad()
        order = torch.randperm(size)
        reshuffle(train_dataloader.dataset)
        for j, idx in enumerate(order.tolist()):
            graph = train_dataloader.dataset[idx]
            N, D = graph.num_nodes, model['paths'].gnn.out_features

            graph_loss, eos_loss, n_uv = 0.0, 0.0, 0
            for u, v in _sample_uv(N):
                ground_truth = graph.paths[u, v]
                ground_truth = ground_truth[ground_truth >= 0]
                if ground_truth.numel() < 2:
                    continue

                count = 1.0
                agt_preds = [u]
                out, hop_targets, visited = [], [], set()
                graph.x = Normal(loc=0.0, scale=1.0).sample((N, D)).to(device)
                graph.x[v] = Normal(loc=-5.0, scale=1.0).sample((1, D)).to(device)
                
                # Teacher forcing: mark the TRUE predecessor 
                # gt[count-1] each step, supervise gt[1:T+1].
                while not (agt_preds[-1] == v or len(agt_preds) > ground_truth.shape[0] - 1):
                    c = int(ground_truth[int(count) - 1])
                    graph.x[c] = Normal(loc=count, scale=1.0).sample((1, D)).to(device)
                    allowed = _allowed_mask(graph.adj[c], visited)
                    if not bool(allowed.any()):
                        break
                    logits = model['paths'](graph).T
                    logits = logits.masked_fill(~allowed.unsqueeze(0), float('-inf'))
                    out.append(logits)
                    agt_preds.append(int(logits.argmax(dim=1)))
                    if nav_cfg['optimal_hop_set']:
                        hop_targets.append((c, v))
                    visited.add(c)
                    count += 1.0

                if not out:
                    continue
                out = torch.cat(out).to(device)
                targets = ground_truth[1:out.shape[0] + 1]

                # Cross-Entropy against the uniform distribution over optimal next-hops.
                if nav_cfg['optimal_hop_set']:
                    soft = out.new_zeros(out.shape)
                    for t, (c, vv) in enumerate(hop_targets[:out.shape[0]]):
                        # Keep only LEGAL (finite) optimal hops: a near-zero-weight edge can
                        # sneak an already-visited node (e.g. the predecessor) into the optimal
                        # set, and soft mass on that masked -inf class makes the loss inf. Fall
                        # back to the canonical successor if none survive.
                        hops = [h for h in _optimal_next_hops(graph, c, vv, nav_cfg['opt_tol'])
                                if torch.isfinite(out[t, h])] or [int(targets[t])]
                        soft[t, torch.as_tensor(hops, device=out.device)] = 1.0 / len(hops)
                    logp = torch.nn.functional.log_softmax(out, dim=1).masked_fill(soft == 0, 0.0)
                    graph_loss = graph_loss + (-(soft * logp).sum(dim=1)).mean()
                else:
                    graph_loss = graph_loss + loss_fn['paths'](out, targets)

                # Reinforce arrival (the final step should select v).
                if nav_cfg['eos_supervision']:
                    eos_loss = eos_loss + torch.nn.functional.cross_entropy(
                        out[-1:], targets[-1:].view(1)
                    )
                n_uv += 1

            if n_uv == 0:
                continue
            loss = {'paths': graph_loss / n_uv}
            total = loss['paths']
            if nav_cfg['eos_supervision']:
                total = total + nav_cfg['eos_weight'] * (eos_loss / n_uv)
            
            # Co-train the detector only when the trunk is not isolated.
            if not nav_cfg['freeze_trunk']:
                preds_edges = torch.stack([
                    model['edges'](graph, graph.edges_x[0, k], graph.edges_x[1, k])
                    for k in range(graph.edges_x.shape[1])
                ]).squeeze(-1).to(device)
                loss['edges'] = loss_fn['edges'](preds_edges, graph.edges_y)
                total = total + loss['edges']

            # Backpropagation.
            (total / batch_size).backward()
            model['paths'].invalidate_cache()
            if not nav_cfg['freeze_trunk']:
                model['edges'].invalidate_cache()

            # Optimization + logging.
            if (j + 1) % batch_size == 0:
                grad_log = _grad_norm_log(trained, prefix='train')
                for module in trained.values():
                    clip_grad_norm_(module.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

                global_step += 1
                wandb.log({
                    'train/loss_paths': loss['paths'].item(),
                    **({'train/loss_edges': loss['edges'].item()} if 'edges' in loss else {}),
                    **({'train/loss_eos': (eos_loss / n_uv).item()} 
                       if nav_cfg['eos_supervision'] else {}),
                    'train/lr_head': optimizer.param_groups[0]['lr'],
                    'epoch': i, 'global_step': global_step,
                    **grad_log,
                })
                print(f"Loss #1: {loss['paths'].item():>7f}  [{j:>5d}/{size:>5d}]")
                print(f"Loss #2: {loss['edges'].item():>7f}  [{j:>5d}/{size:>5d}]"
                        if 'edges' in loss else "")

    # Early-stopping restore.
    if best_state:
        model['paths'].load_state_dict(best_state['paths'])
        model['edges'].load_state_dict(best_state['edges'])

    # Final test.
    test_loop_paths(test_dataloader, model, loss_fn, wandb_prefix='test')
    run.finish()


# Cross-Entropy for autoregressive path generation; BCE kept for detector monitoring.
loss_fn = {'paths': nn.CrossEntropyLoss(), 'edges': nn.BCEWithLogitsLoss()}
train_dataloader = DataLoader(train_graphs, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs, batch_size=batch_size)
test_dataloader = DataLoader(test_graphs, batch_size=batch_size)
if train_paths:
    if nav_cfg['freeze_trunk']:
        for p in navigator.gnn.parameters():
            p.requires_grad_(False)
        optimizer = torch.optim.AdamW([
            {'params': navigator.head.parameters(), 'lr': 3e-4},
            {'params': navigator.classifier.parameters(), 'lr': 3e-4},
        ], betas=(0.9, 0.95), weight_decay=0.05)
    else:
        for p in navigator.gnn.parameters():
            p.requires_grad_(True)
        optimizer = torch.optim.AdamW([
            {'params': navigator.gnn.parameters(), 'lr': 3e-5},
            {'params': navigator.head.parameters(), 'lr': 3e-4},
            {'params': navigator.classifier.parameters(), 'lr': 3e-4},
            {'params': detector.classifier.parameters(), 'lr': 3e-4, 'weight_decay': 0.0},
        ], betas=(0.9, 0.95), weight_decay=0.05)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    train_loop_paths(train_dataloader, val_dataloader, test_dataloader,
                     {'paths': navigator, 'edges': detector}, loss_fn, optimizer,
                     scheduler, batch_size=batch_size, epochs=epochs)
    del optimizer, scheduler
    gc.collect()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/arushar/.netrc.
wandb: Currently logged in as: arushar (alelab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Validation #1
Test Error #1: 
 Success: 19.0% | SPL: 0.086 | Valid: 100.0% 
 Reached goal: 19.0% | Optimal (hops): 3.572 | Exact (hops):  4.1% 
 Avg loss: 3.4572 

Test Error #2: 
 Accuracy: 98.4% | F1: 0.984 | P: 0.972 | R: 0.996 | Bal Acc: 98.4% | Avg loss: 0.052319

Epoch #1
Loss #1: 1.756048  [    3/   32]
Loss #2: 0.033042  [    3/   32]
Loss #1: 1.760902  [    7/   32]
Loss #2: 0.014842  [    7/   32]
Loss #1: 1.060889  [   11/   32]
Loss #2: 0.019936  [   11/   32]
Loss #1: 1.302350  [   15/   32]
Loss #2: 0.050586  [   15/   32]
Loss #1: 1.424085  [   19/   32]
Loss #2: 0.089339  [   19/   32]
Loss #1: 1.107597  [   23/   32]
Loss #2: 0.010544  [   23/   32]
Loss #1: 1.339374  [   27/   32]
Loss #2: 0.127444  [   27/   32]
Loss #1: 0.963958  [   31/   32]
Loss #2: 0.013693  [   31/   32]
Epoch #2
Loss #1: 1.162775  [    3/   32]
Loss #2: 0.251446  [    3/   32]
Loss #1: 1.040993  [    7/   32]
Loss #2: 0.036835  [    7/   32]
Loss #1: 0.997578  [   11/   32]
Loss #2: 0.024171  

In [ ]:
if train_paths:
    torch.save(navigator, '../outputs/e9_multistage_training/path_navigator.pt')
    torch.save(navigator.gnn.state_dict(), f'../outputs/e9_multistage_training/path_navigator_{model_type}.pt')
else:
    navigator = torch.load('../outputs/e9_multistage_training/suite1/path_navigator.pt', weights_only=False)
    gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/suite1/path_navigator_{model_type}.pt'))

#### Evaluation of Fine-Tuned GNN on Shortest Paths
We thus test the fine-tuned model on the evaluation dataset. First, we we will render the output for clarity.

In [ ]:
# Test out the Navigator.
navigator.eval().to(device)
with torch.no_grad():
    out = navigator(ex_graph)

render_matrix(out.softmax(dim=0).T)

Matrix([[0.0521, 0.0306, 0.00965, 0.0564, 0.0252, 0.0122, 0.0218, 0.0399, 0.0306, 0.0415, 0.0242, 0.0746, 0.0122, 0.0155, 0.0371, 0.0444, 0.0319, 0.00758, 0.0151, 0.063, 0.0109, 0.0193, 0.0133, 0.0124, 0.0128, 0.0221, 0.0649, 0.0216, 0.0779, 0.0228, 0.0118, 0.043, 0.0217]])

In [ ]:
# Test out the Navigator's generation abilities.
N = ex_graph.num_nodes
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
with torch.no_grad():
    out = navigator.generate(ex_graph, node1, node2)

print(node1, node2)
print(ex_graph.paths[node1, node2].tolist())
render_matrix(out.T, sig_figs=0)

4 14
[14, 11, 12, 19, 18, 22, 4, 0, 0]


Matrix([[4, 22, 23, 26, 27, 25, 32, 31, 30, 7]])

In [ ]:
# Evaluate the GNN on its reconstruction of test graph shortest paths.
test_loop_paths(test_dataloader, {'paths': navigator, 'edges': detector}, loss_fn)
pass

Test Error #1: 
 Success: 27.2% | SPL: 0.146 | Valid: 100.0% 
 Reached goal: 27.2% | Optimal (hops): 3.394 | Exact (hops):  6.8% 
 Avg loss: 3.5347 

Test Error #2: 
 Accuracy: 94.4% | F1: 0.945 | P: 0.905 | R: 0.989 | Bal Acc: 94.3% | Avg loss: 0.202691



In [ ]:
# Evaluate the GNN on its reconstruction of test graph shortest path distances.
spd_gnn.gnn.load_state_dict(navigator.gnn.state_dict())
test_loop_dists(
    test_dataloader, {'dists': spd_gnn, 'edges': detector},
    {'dists': nn.MSELoss(), 'edges': nn.BCEWithLogitsLoss()}
)
pass

Test Error #1: 
 Avg error: 2.662 
 Avg loss: 1306.902853 

Test Error #2: 
 Accuracy: 93.7%, F1: 0.940 | P: 0.898 | R: 0.986 | Bal Acc: 93.7% | Avg loss: 0.225430 

